# 03 Model Training and Validation

This notebook trains and validates regression models for the capstone project:

**A Data-Driven Framework for Prioritizing Public Capital Allocation to Basic Education Infrastructure across Nigerian States Using Education Need Indicators**

## Purpose

This notebook performs the following tasks:

1. Loads the BENI-Core and BENI-Expanded model-ready datasets.
2. Trains four regression models:
   - Linear Regression
   - Ridge Regression
   - Random Forest Regressor
   - Gradient Boosting Regressor
3. Evaluates BENI-Core on a 2022 holdout test set.
4. Evaluates BENI-Expanded using 5-fold cross-validation.
5. Produces ranking validation outputs.
6. Exports model-performance and ranking-comparison tables.

## Interpretation Rule

The modelling task validates BENI reproducibility and ranking stability. It should not be interpreted as causal prediction because BENI is an engineered composite index derived from the selected indicators.


In [ ]:
# Import required libraries.
from pathlib import Path
import pandas as pd
import numpy as np

# Import modelling tools.
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

# Configure pandas display.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Define project paths.
PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
MODEL_RESULTS_DIR = PROJECT_ROOT / "outputs" / "model_results"

# Create output directories.
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Define reproducibility seed.
RANDOM_STATE = 42


## 1. Load Model-Ready Datasets

The model-ready files are expected in `data/processed/`.


In [ ]:
# Load BENI-Core model-ready dataset.
core_df = pd.read_csv(PROCESSED_DIR / "beni_core_model_ready.csv")

# Load BENI-Expanded model-ready dataset.
expanded_df = pd.read_csv(PROCESSED_DIR / "beni_expanded_2022_model_ready.csv")

# Confirm shapes.
print("BENI-Core dataset shape:", core_df.shape)
print("BENI-Expanded dataset shape:", expanded_df.shape)


## 2. Define Features, Targets, and Models

The selected predictors match the report methodology.

BENI-Core uses five UBEC-derived indicators. BENI-Expanded uses the same five indicators plus the NBS socioeconomic need proxy.


In [ ]:
# Define BENI-Core features.
core_features = [
    "pupil_teacher_ratio",
    "pupil_classroom_ratio",
    "unqualified_teacher_share",
    "bad_classroom_share",
    "enrolment_per_school",
]

# Define BENI-Expanded features.
expanded_features = core_features + [
    "nbs_socioeconomic_need_proxy",
]

# Define target variables.
core_target = "target_beni_core_score"
expanded_target = "target_beni_expanded_score"

# Define models.
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),
}

# Display feature summary.
print("Core features:", core_features)
print("Expanded features:", expanded_features)


## 3. BENI-Core Holdout Validation

BENI-Core is trained on 2018, 2019, and 2020 records and tested on the 2022 holdout set.


In [ ]:
# Split BENI-Core dataset into training and test sets using the split_group field.
core_train = core_df[core_df["split_group"].str.contains("TRAIN", case=False, na=False)].copy()
core_test = core_df[core_df["split_group"].str.contains("TEST", case=False, na=False)].copy()

# If split labels differ, use year-based fallback.
if core_train.empty or core_test.empty:
    core_train = core_df[core_df["year"].isin([2018, 2019, 2020])].copy()
    core_test = core_df[core_df["year"] == 2022].copy()

# Define X and y.
X_train_core = core_train[core_features]
y_train_core = core_train[core_target]
X_test_core = core_test[core_features]
y_test_core = core_test[core_target]

# Confirm split sizes.
print("Core train rows:", X_train_core.shape[0])
print("Core test rows:", X_test_core.shape[0])


In [ ]:
# Evaluate BENI-Core models on the 2022 holdout test set.
core_results = []
core_predictions = core_test[["state", "year", core_target, "beni_rank_within_year"]].copy()

for model_name, model in models.items():
    # Fit model.
    model.fit(X_train_core, y_train_core)

    # Predict on holdout test set.
    y_pred = model.predict(X_test_core)

    # Calculate metrics.
    mae = mean_absolute_error(y_test_core, y_pred)
    rmse = mean_squared_error(y_test_core, y_pred, squared=False)
    r2 = r2_score(y_test_core, y_pred)
    spearman_corr, spearman_p = spearmanr(y_test_core, y_pred)

    # Store results.
    core_results.append({
        "Model": model_name,
        "Training Rows": X_train_core.shape[0],
        "Test Rows": X_test_core.shape[0],
        "Features": len(core_features),
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "Spearman Rank Correlation": spearman_corr,
        "Spearman p-value": spearman_p,
    })

    # Store predictions.
    safe_model_name = model_name.lower().replace(" ", "_").replace("-", "_")
    core_predictions[f"{safe_model_name}_prediction"] = y_pred

# Create results dataframe.
core_model_performance = pd.DataFrame(core_results)

# Save outputs.
core_model_performance.to_csv(TABLES_DIR / "core_model_performance.csv", index=False)
core_predictions.to_csv(TABLES_DIR / "core_model_predictions.csv", index=False)

# Display results.
core_model_performance


## 4. BENI-Expanded 5-Fold Cross-Validation

BENI-Expanded has 37 state-level records for 2022. Therefore, 5-fold cross-validation is used.


In [ ]:
# Define expanded X and y.
X_expanded = expanded_df[expanded_features]
y_expanded = expanded_df[expanded_target]

# Define 5-fold cross-validation.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Confirm dataset size.
print("Expanded rows:", X_expanded.shape[0])
print("Expanded features:", X_expanded.shape[1])


In [ ]:
# Evaluate BENI-Expanded models using 5-fold cross-validation.
expanded_results = []
expanded_predictions = expanded_df[
    ["state", "year", expanded_target, "beni_expanded_rank"]
].copy()

for model_name, model in models.items():
    # Generate cross-validated predictions.
    y_pred_cv = cross_val_predict(model, X_expanded, y_expanded, cv=cv)

    # Calculate metrics.
    mae = mean_absolute_error(y_expanded, y_pred_cv)
    rmse = mean_squared_error(y_expanded, y_pred_cv, squared=False)
    r2 = r2_score(y_expanded, y_pred_cv)
    spearman_corr, spearman_p = spearmanr(y_expanded, y_pred_cv)

    # Store results.
    expanded_results.append({
        "Model": model_name,
        "Validation Method": "5-fold cross-validation",
        "Rows": X_expanded.shape[0],
        "Features": len(expanded_features),
        "Mean MAE": mae,
        "Mean RMSE": rmse,
        "Mean R²": r2,
        "Spearman Rank Correlation": spearman_corr,
        "Spearman p-value": spearman_p,
    })

    # Store predictions.
    safe_model_name = model_name.lower().replace(" ", "_").replace("-", "_")
    expanded_predictions[f"{safe_model_name}_prediction"] = y_pred_cv

# Create results dataframe.
expanded_model_performance = pd.DataFrame(expanded_results)

# Save outputs.
expanded_model_performance.to_csv(TABLES_DIR / "expanded_model_performance.csv", index=False)
expanded_predictions.to_csv(TABLES_DIR / "expanded_model_predictions.csv", index=False)

# Display results.
expanded_model_performance


## 5. Ranking Validation for Best Expanded Model

The report uses Linear Regression as the main interpretability model because it best reproduces BENI scores and provides coefficient interpretation.


In [ ]:
# Fit Linear Regression on the full expanded dataset for final ranking validation.
best_expanded_model = LinearRegression()
best_expanded_model.fit(X_expanded, y_expanded)

# Predict BENI-Expanded scores.
expanded_ranking_comparison = expanded_df[
    ["state", "year", expanded_target, "beni_expanded_rank"]
].copy()

expanded_ranking_comparison["ml_predicted_beni_expanded_score"] = best_expanded_model.predict(X_expanded)

# Rank predicted scores: highest need score receives rank 1.
expanded_ranking_comparison["ml_predicted_beni_expanded_rank"] = (
    expanded_ranking_comparison["ml_predicted_beni_expanded_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Calculate rank difference.
expanded_ranking_comparison["rank_difference"] = (
    expanded_ranking_comparison["ml_predicted_beni_expanded_rank"]
    - expanded_ranking_comparison["beni_expanded_rank"]
)

# Sort by actual BENI-Expanded rank.
expanded_ranking_comparison = expanded_ranking_comparison.sort_values("beni_expanded_rank").reset_index(drop=True)

# Calculate top-10 overlap.
actual_top_10 = set(expanded_ranking_comparison.nsmallest(10, "beni_expanded_rank")["state"])
predicted_top_10 = set(expanded_ranking_comparison.nsmallest(10, "ml_predicted_beni_expanded_rank")["state"])
top_10_overlap_count = len(actual_top_10.intersection(predicted_top_10))
top_10_overlap_rate = top_10_overlap_count / 10

# Calculate Spearman correlation.
rank_corr, rank_p = spearmanr(
    expanded_ranking_comparison["beni_expanded_rank"],
    expanded_ranking_comparison["ml_predicted_beni_expanded_rank"]
)

# Save ranking comparison output.
expanded_ranking_comparison.to_csv(TABLES_DIR / "expanded_beni_ranking_comparison.csv", index=False)

# Display validation summary.
print("Spearman rank correlation:", round(rank_corr, 6))
print("Spearman p-value:", round(rank_p, 6))
print("Top-10 overlap count:", top_10_overlap_count)
print("Top-10 overlap rate:", f"{top_10_overlap_rate:.2%}")

expanded_ranking_comparison.head(10)


## 6. Coefficient Importance for BENI-Expanded

Linear Regression coefficients are used to explain the contribution of each predictor within the engineered BENI-Expanded framework.


In [ ]:
# Create coefficient-importance table for the best expanded model.
coefficient_importance = pd.DataFrame({
    "feature": expanded_features,
    "coefficient": best_expanded_model.coef_,
})

# Add absolute coefficient and ranking.
coefficient_importance["absolute_coefficient"] = coefficient_importance["coefficient"].abs()
coefficient_importance = coefficient_importance.sort_values(
    "absolute_coefficient", ascending=False
).reset_index(drop=True)
coefficient_importance["rank"] = coefficient_importance.index + 1

# Reorder columns.
coefficient_importance = coefficient_importance[
    ["rank", "feature", "coefficient", "absolute_coefficient"]
]

# Save output.
coefficient_importance.to_csv(TABLES_DIR / "coefficient_importance_beni_expanded.csv", index=False)

# Display table.
coefficient_importance


## 7. Export Top 10 Priority States

This output supports the report’s preliminary results and Appendix C.


In [ ]:
# Extract top 10 priority states from ranking comparison.
top_10_priority_states = expanded_ranking_comparison.head(10).copy()

# Rename columns for report readability.
top_10_priority_states = top_10_priority_states.rename(columns={
    expanded_target: "actual_beni_expanded_score",
    "beni_expanded_rank": "actual_beni_expanded_rank",
})

# Save output.
top_10_priority_states.to_csv(TABLES_DIR / "top_10_priority_states.csv", index=False)

# Display top 10.
top_10_priority_states


## 8. Create Model Summary Table

This table consolidates the most important model-validation outputs for documentation and reporting.


In [ ]:
# Identify best core and expanded models by lowest RMSE.
best_core_row = core_model_performance.sort_values("RMSE").iloc[0]
best_expanded_row = expanded_model_performance.sort_values("Mean RMSE").iloc[0]

# Create model summary.
model_summary = pd.DataFrame([
    {
        "dataset": "BENI-Core",
        "target": core_target,
        "best_model": best_core_row["Model"],
        "validation_strategy": "2018-2020 train; 2022 holdout test",
        "rows": len(core_df),
        "features": len(core_features),
        "best_rmse": best_core_row["RMSE"],
        "best_r2": best_core_row["R²"],
        "best_spearman": best_core_row["Spearman Rank Correlation"],
        "top_10_overlap_rate": np.nan,
    },
    {
        "dataset": "BENI-Expanded",
        "target": expanded_target,
        "best_model": best_expanded_row["Model"],
        "validation_strategy": "5-fold cross-validation and full-sample ranking validation",
        "rows": len(expanded_df),
        "features": len(expanded_features),
        "best_rmse": best_expanded_row["Mean RMSE"],
        "best_r2": best_expanded_row["Mean R²"],
        "best_spearman": rank_corr,
        "top_10_overlap_rate": top_10_overlap_rate,
    },
])

# Save output.
model_summary.to_csv(TABLES_DIR / "model_summary.csv", index=False)

# Display model summary.
model_summary


## 9. Export Consolidated Model Workbook

All key model outputs are exported into one Excel workbook for repository evidence.


In [ ]:
# Define workbook output path.
model_workbook_path = MODEL_RESULTS_DIR / "Capstone_ML_Model_Results_Workbook.xlsx"

# Export all core model outputs into a single workbook.
with pd.ExcelWriter(model_workbook_path, engine="openpyxl") as writer:
    model_summary.to_excel(writer, sheet_name="Model Summary", index=False)
    core_model_performance.to_excel(writer, sheet_name="Core Model Performance", index=False)
    expanded_model_performance.to_excel(writer, sheet_name="Expanded Model Performance", index=False)
    expanded_ranking_comparison.to_excel(writer, sheet_name="Expanded Ranking", index=False)
    top_10_priority_states.to_excel(writer, sheet_name="Top 10 Priority States", index=False)
    coefficient_importance.to_excel(writer, sheet_name="Coefficient Importance", index=False)

print("Model workbook created:", model_workbook_path)


## 10. Final Output Checklist

Expected files:

```text
outputs/tables/core_model_performance.csv
outputs/tables/core_model_predictions.csv
outputs/tables/expanded_model_performance.csv
outputs/tables/expanded_model_predictions.csv
outputs/tables/expanded_beni_ranking_comparison.csv
outputs/tables/coefficient_importance_beni_expanded.csv
outputs/tables/top_10_priority_states.csv
outputs/tables/model_summary.csv
outputs/model_results/Capstone_ML_Model_Results_Workbook.xlsx
```


In [ ]:
# Confirm exported files.
expected_files = [
    TABLES_DIR / "core_model_performance.csv",
    TABLES_DIR / "core_model_predictions.csv",
    TABLES_DIR / "expanded_model_performance.csv",
    TABLES_DIR / "expanded_model_predictions.csv",
    TABLES_DIR / "expanded_beni_ranking_comparison.csv",
    TABLES_DIR / "coefficient_importance_beni_expanded.csv",
    TABLES_DIR / "top_10_priority_states.csv",
    TABLES_DIR / "model_summary.csv",
    MODEL_RESULTS_DIR / "Capstone_ML_Model_Results_Workbook.xlsx",
]

for file_path in expected_files:
    print(f"{file_path}: {'FOUND' if file_path.exists() else 'MISSING'}")
